# Machine Translation System: English ↔ Marathi

This notebook implements a comprehensive Machine Translation system for translating public information content between English and Marathi.

## Features:
- **Bidirectional Translation**: English → Marathi and Marathi → English
- **Multiple Translation Approaches**: API-based, Transformer-based, and Seq2Seq models
- **BLEU Score Evaluation**: Measure translation quality
- **Public Information Focus**: News, announcements, documents
- **Interactive Interface**: Easy-to-use translation interface

## 1. Installation and Setup

In [ ]:
# Install required packages
!pip install transformers -q
!pip install sentencepiece -q
!pip install sacrebleu -q
!pip install indic-nlp-library -q
!pip install datasets -q
!pip install googletrans==3.1.0a0 -q
!pip install torch -q
!pip install nltk -q
!pip install indicnlp -q

print("✅ All packages installed successfully!")

In [ ]:
# Import libraries
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from typing import List, Dict, Tuple
import re
import json

# NLP libraries
import nltk
from nltk.tokenize import word_tokenize, sent_tokenize
from nltk.translate.bleu_score import sentence_bleu, corpus_bleu, SmoothingFunction

# Transformers
from transformers import (
    AutoTokenizer, 
    AutoModelForSeq2SeqLM,
    MarianMTModel,
    MarianTokenizer,
    pipeline
)

# Google Translate API
from googletrans import Translator

# PyTorch
import torch
from torch import nn
import torch.nn.functional as F

# Download NLTK data
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)

# Set device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")
print("\n✅ Setup complete!")

## 2. Google Translate API Approach

Quick and easy translation using Google Translate API.

In [ ]:
class GoogleTranslateSystem:
    """
    Machine Translation using Google Translate API.
    """
    
    def __init__(self):
        self.translator = Translator()
        
    def translate_en_to_mr(self, text: str) -> str:
        """
        Translate English to Marathi.
        """
        try:
            translation = self.translator.translate(text, src='en', dest='mr')
            return translation.text
        except Exception as e:
            print(f"Translation error: {e}")
            return text
    
    def translate_mr_to_en(self, text: str) -> str:
        """
        Translate Marathi to English.
        """
        try:
            translation = self.translator.translate(text, src='mr', dest='en')
            return translation.text
        except Exception as e:
            print(f"Translation error: {e}")
            return text
    
    def translate_batch(self, texts: List[str], src: str = 'en', dest: str = 'mr') -> List[str]:
        """
        Translate multiple texts.
        """
        translations = []
        for text in texts:
            try:
                result = self.translator.translate(text, src=src, dest=dest)
                translations.append(result.text)
            except:
                translations.append(text)
        return translations

# Initialize
google_mt = GoogleTranslateSystem()
print("✅ Google Translate system initialized!")

In [ ]:
# Test Google Translate
print("=" * 80)
print("Google Translate Demo")
print("=" * 80)

# English to Marathi
en_text = "The government has announced new policies for public welfare."
mr_translation = google_mt.translate_en_to_mr(en_text)

print(f"\n🇬🇧 English: {en_text}")
print(f"🇮🇳 Marathi: {mr_translation}")

# Marathi to English
mr_text = "आज मुंबईमध्ये मोठा कार्यक्रम आयोजित करण्यात आला आहे."
en_translation = google_mt.translate_mr_to_en(mr_text)

print(f"\n🇮🇳 Marathi: {mr_text}")
print(f"🇬🇧 English: {en_translation}")

## 3. Transformer-based Translation (IndicTrans2)

Using state-of-the-art transformer models specifically trained for Indian languages.

In [ ]:
# Install IndicTrans2
!git clone https://github.com/AI4Bharat/IndicTrans2.git
%cd IndicTrans2
!pip install -q .
%cd ..

In [ ]:
class IndicTransTranslator:
    """
    Translation using AI4Bharat's IndicTrans2 model.
    Specifically designed for Indian languages.
    """
    
    def __init__(self):
        print("Loading IndicTrans2 models...")
        
        # Load English to Indic model
        self.en_indic_model_name = "ai4bharat/indictrans2-en-indic-1B"
        self.en_indic_tokenizer = AutoTokenizer.from_pretrained(
            self.en_indic_model_name, 
            trust_remote_code=True
        )
        self.en_indic_model = AutoModelForSeq2SeqLM.from_pretrained(
            self.en_indic_model_name, 
            trust_remote_code=True
        ).to(device)
        
        # Load Indic to English model
        self.indic_en_model_name = "ai4bharat/indictrans2-indic-en-1B"
        self.indic_en_tokenizer = AutoTokenizer.from_pretrained(
            self.indic_en_model_name, 
            trust_remote_code=True
        )
        self.indic_en_model = AutoModelForSeq2SeqLM.from_pretrained(
            self.indic_en_model_name, 
            trust_remote_code=True
        ).to(device)
        
        print("✅ Models loaded successfully!")
    
    def translate_en_to_mr(self, text: str, max_length: int = 256) -> str:
        """
        Translate English to Marathi.
        """
        # Add language tag for Marathi
        input_text = f"<2mr> {text}"
        
        # Tokenize
        inputs = self.en_indic_tokenizer(
            input_text, 
            return_tensors="pt", 
            padding=True,
            truncation=True,
            max_length=max_length
        ).to(device)
        
        # Generate translation
        with torch.no_grad():
            outputs = self.en_indic_model.generate(
                **inputs,
                max_length=max_length,
                num_beams=5,
                num_return_sequences=1,
                early_stopping=True
            )
        
        # Decode
        translation = self.en_indic_tokenizer.decode(
            outputs[0], 
            skip_special_tokens=True
        )
        
        return translation
    
    def translate_mr_to_en(self, text: str, max_length: int = 256) -> str:
        """
        Translate Marathi to English.
        """
        # Add language tag for Marathi
        input_text = f"<2en> {text}"
        
        # Tokenize
        inputs = self.indic_en_tokenizer(
            input_text, 
            return_tensors="pt", 
            padding=True,
            truncation=True,
            max_length=max_length
        ).to(device)
        
        # Generate translation
        with torch.no_grad():
            outputs = self.indic_en_model.generate(
                **inputs,
                max_length=max_length,
                num_beams=5,
                num_return_sequences=1,
                early_stopping=True
            )
        
        # Decode
        translation = self.indic_en_tokenizer.decode(
            outputs[0], 
            skip_special_tokens=True
        )
        
        return translation

# Initialize (this may take a few minutes)
print("Initializing IndicTrans2 translator...")
print("⏳ This may take a few minutes...")
try:
    indic_mt = IndicTransTranslator()
except Exception as e:
    print(f"Note: IndicTrans2 loading failed. Using Google Translate as fallback.")
    print(f"Error: {e}")
    indic_mt = None

## 4. Comprehensive Translation System

Combined system with multiple backends and fallback options.

In [ ]:
class MarathiTranslationSystem:
    """
    Comprehensive English-Marathi translation system.
    Supports multiple backends with automatic fallback.
    """
    
    def __init__(self, backend='google'):
        """
        Initialize translation system.
        
        Args:
            backend: 'google' or 'indictrans'
        """
        self.backend = backend
        self.google_translator = GoogleTranslateSystem()
        self.indic_translator = indic_mt if indic_mt else None
        
        print(f"✅ Translation system initialized with backend: {backend}")
    
    def translate(self, text: str, direction: str = 'en-mr') -> str:
        """
        Translate text.
        
        Args:
            text: Input text
            direction: 'en-mr' or 'mr-en'
        """
        if self.backend == 'indictrans' and self.indic_translator:
            try:
                if direction == 'en-mr':
                    return self.indic_translator.translate_en_to_mr(text)
                else:
                    return self.indic_translator.translate_mr_to_en(text)
            except:
                print("IndicTrans failed, falling back to Google Translate")
                pass
        
        # Fallback to Google Translate
        if direction == 'en-mr':
            return self.google_translator.translate_en_to_mr(text)
        else:
            return self.google_translator.translate_mr_to_en(text)
    
    def translate_document(self, text: str, direction: str = 'en-mr') -> str:
        """
        Translate a longer document sentence by sentence.
        """
        sentences = sent_tokenize(text)
        translations = []
        
        for sentence in sentences:
            translation = self.translate(sentence.strip(), direction)
            translations.append(translation)
        
        return ' '.join(translations)
    
    def batch_translate(self, texts: List[str], direction: str = 'en-mr') -> List[str]:
        """
        Translate multiple texts.
        """
        return [self.translate(text, direction) for text in texts]

# Initialize translation system
mt_system = MarathiTranslationSystem(backend='google')
print("✅ Ready for translation!")

## 5. Public Information Translation Examples

In [ ]:
# Sample public information content
public_announcements = {
    'government_policy': "The Ministry of Health has announced new healthcare initiatives for rural areas.",
    'public_notice': "All citizens are requested to carry valid identification documents.",
    'weather_alert': "Heavy rainfall is expected in the coastal regions tomorrow.",
    'education': "The education department has released the annual examination schedule.",
    'transportation': "New metro line will be operational from next month."
}

print("=" * 80)
print("PUBLIC INFORMATION TRANSLATION: English → Marathi")
print("=" * 80)

for category, text in public_announcements.items():
    translation = mt_system.translate(text, direction='en-mr')
    
    print(f"\n📢 Category: {category.upper().replace('_', ' ')}")
    print(f"\n🇬🇧 English:")
    print(f"   {text}")
    print(f"\n🇮🇳 Marathi:")
    print(f"   {translation}")
    print("-" * 80)

In [ ]:
# Sample Marathi content (public information)
marathi_content = {
    'news': "महाराष्ट्र सरकारने नवीन धोरण जाहीर केले आहे.",
    'announcement': "सर्व नागरिकांना लसीकरण केंद्रात उपस्थित राहण्याची विनंती.",
    'alert': "उद्या मुंबईत मुसळधार पावसाची शक्यता आहे.",
    'notice': "कृपया आपले आधार कार्ड सोबत घेऊन या."
}

print("=" * 80)
print("PUBLIC INFORMATION TRANSLATION: Marathi → English")
print("=" * 80)

for category, text in marathi_content.items():
    translation = mt_system.translate(text, direction='mr-en')
    
    print(f"\n📢 Category: {category.upper()}")
    print(f"\n🇮🇳 Marathi:")
    print(f"   {text}")
    print(f"\n🇬🇧 English:")
    print(f"   {translation}")
    print("-" * 80)

## 6. Document Translation

In [ ]:
# Long document example
english_document = """
The Government of Maharashtra has announced a new public welfare scheme. 
This scheme aims to provide financial assistance to farmers affected by drought. 
All eligible farmers can apply online through the official government portal. 
The application deadline is 31st December 2024. 
For more information, please visit your nearest government office.
"""

print("=" * 80)
print("DOCUMENT TRANSLATION")
print("=" * 80)

print("\n📄 Original English Document:")
print(english_document)

print("\n⏳ Translating...")
marathi_document = mt_system.translate_document(english_document, direction='en-mr')

print("\n📄 Translated Marathi Document:")
print(marathi_document)

## 7. Translation Quality Evaluation (BLEU Score)

In [ ]:
def calculate_bleu_score(reference: str, candidate: str) -> Dict[str, float]:
    """
    Calculate BLEU score between reference and candidate translations.
    """
    # Tokenize
    reference_tokens = word_tokenize(reference.lower())
    candidate_tokens = word_tokenize(candidate.lower())
    
    # Smoothing function for better scores with short sentences
    smoothing = SmoothingFunction()
    
    # Calculate BLEU scores
    bleu1 = sentence_bleu([reference_tokens], candidate_tokens, 
                          weights=(1, 0, 0, 0),
                          smoothing_function=smoothing.method1)
    bleu2 = sentence_bleu([reference_tokens], candidate_tokens, 
                          weights=(0.5, 0.5, 0, 0),
                          smoothing_function=smoothing.method1)
    bleu3 = sentence_bleu([reference_tokens], candidate_tokens, 
                          weights=(0.33, 0.33, 0.33, 0),
                          smoothing_function=smoothing.method1)
    bleu4 = sentence_bleu([reference_tokens], candidate_tokens, 
                          weights=(0.25, 0.25, 0.25, 0.25),
                          smoothing_function=smoothing.method1)
    
    return {
        'BLEU-1': bleu1,
        'BLEU-2': bleu2,
        'BLEU-3': bleu3,
        'BLEU-4': bleu4
    }

def evaluate_translation_quality(test_cases: List[Dict]) -> pd.DataFrame:
    """
    Evaluate translation quality using BLEU scores.
    """
    results = []
    
    for case in test_cases:
        source = case['source']
        reference = case['reference']
        direction = case.get('direction', 'en-mr')
        
        # Get translation
        candidate = mt_system.translate(source, direction=direction)
        
        # Calculate BLEU scores
        scores = calculate_bleu_score(reference, candidate)
        
        results.append({
            'Source': source[:50] + '...' if len(source) > 50 else source,
            'Reference': reference[:50] + '...' if len(reference) > 50 else reference,
            'Translation': candidate[:50] + '...' if len(candidate) > 50 else candidate,
            **scores
        })
    
    return pd.DataFrame(results)

print("✅ Evaluation functions ready!")

In [ ]:
# Test cases with reference translations
test_cases = [
    {
        'source': 'Good morning',
        'reference': 'सुप्रभात',
        'direction': 'en-mr'
    },
    {
        'source': 'Thank you',
        'reference': 'धन्यवाद',
        'direction': 'en-mr'
    },
    {
        'source': 'The government has announced new policies.',
        'reference': 'सरकारने नवीन धोरणे जाहीर केली आहेत.',
        'direction': 'en-mr'
    },
    {
        'source': 'Please maintain social distancing.',
        'reference': 'कृपया सामाजिक अंतर राखा.',
        'direction': 'en-mr'
    },
    {
        'source': 'Education is very important.',
        'reference': 'शिक्षण अत्यंत महत्त्वाचे आहे.',
        'direction': 'en-mr'
    }
]

print("=" * 80)
print("TRANSLATION QUALITY EVALUATION")
print("=" * 80)

results_df = evaluate_translation_quality(test_cases)
display(results_df)

# Calculate average scores
print("\n📊 Average BLEU Scores:")
print("-" * 40)
for col in ['BLEU-1', 'BLEU-2', 'BLEU-3', 'BLEU-4']:
    avg_score = results_df[col].mean()
    print(f"{col}: {avg_score:.4f}")

## 8. Visualization of BLEU Scores

In [ ]:
# Plot BLEU scores
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Bar plot
bleu_columns = ['BLEU-1', 'BLEU-2', 'BLEU-3', 'BLEU-4']
avg_scores = [results_df[col].mean() for col in bleu_columns]

axes[0].bar(bleu_columns, avg_scores, color=['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728'])
axes[0].set_ylabel('Score', fontsize=12)
axes[0].set_title('Average BLEU Scores', fontsize=14, fontweight='bold')
axes[0].set_ylim(0, 1)
axes[0].grid(axis='y', alpha=0.3)

for i, score in enumerate(avg_scores):
    axes[0].text(i, score + 0.02, f'{score:.3f}', ha='center', fontweight='bold')

# Heatmap
sns.heatmap(results_df[bleu_columns].T, annot=True, fmt='.3f', 
            cmap='YlGnBu', ax=axes[1], cbar_kws={'label': 'BLEU Score'})
axes[1].set_xlabel('Test Case', fontsize=12)
axes[1].set_ylabel('BLEU Score Type', fontsize=12)
axes[1].set_title('BLEU Scores Heatmap', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.show()

## 9. Interactive Translation Interface

In [ ]:
def interactive_translator():
    """
    Interactive translation interface.
    """
    print("="  * 80)
    print("🌐 INTERACTIVE ENGLISH-MARATHI TRANSLATOR")
    print("=" * 80)
    print("\nOptions:")
    print("1. English → Marathi")
    print("2. Marathi → English")
    print("3. Exit")
    print("-" * 80)
    
    while True:
        choice = input("\nSelect option (1/2/3): ").strip()
        
        if choice == '3':
            print("\n👋 Thank you for using the translator!")
            break
        
        if choice not in ['1', '2']:
            print("❌ Invalid choice. Please select 1, 2, or 3.")
            continue
        
        text = input("\nEnter text to translate: ").strip()
        
        if not text:
            print("❌ Empty text. Please enter some text.")
            continue
        
        direction = 'en-mr' if choice == '1' else 'mr-en'
        
        print("\n⏳ Translating...")
        translation = mt_system.translate(text, direction=direction)
        
        print("\n" + "=" * 80)
        print(f"📝 Original ({('English' if choice == '1' else 'Marathi')}):")
        print(f"   {text}")
        print(f"\n✅ Translation ({('Marathi' if choice == '1' else 'English')}):")
        print(f"   {translation}")
        print("=" * 80)

# Run interactive translator
# Uncomment the line below to run
# interactive_translator()

## 10. Batch Translation for Public Documents

In [ ]:
# Sample public information bulletins
public_bulletins = [
    "Vaccination drive will be conducted in all districts from Monday.",
    "Citizens are advised to follow COVID-19 safety guidelines.",
    "New traffic rules will be implemented from next week.",
    "Free health checkup camps are organized in rural areas.",
    "Online application portal is now available for ration cards.",
    "Emergency helpline number: 1077 available 24/7."
]

print("=" * 80)
print("BATCH TRANSLATION: Public Information Bulletins")
print("=" * 80)

# Translate all bulletins
marathi_bulletins = mt_system.batch_translate(public_bulletins, direction='en-mr')

# Create DataFrame
bulletin_df = pd.DataFrame({
    'English': public_bulletins,
    'Marathi': marathi_bulletins
})

display(bulletin_df)

print("\n✅ All bulletins translated successfully!")

## 11. Save Translations to File

In [ ]:
def save_translations(translations_df: pd.DataFrame, filename: str = 'translations.csv'):
    """
    Save translations to CSV file.
    """
    translations_df.to_csv(filename, index=False, encoding='utf-8-sig')
    print(f"✅ Translations saved to {filename}")

def save_translation_report(results_df: pd.DataFrame, filename: str = 'translation_report.txt'):
    """
    Save detailed translation report.
    """
    with open(filename, 'w', encoding='utf-8') as f:
        f.write("MACHINE TRANSLATION EVALUATION REPORT\n")
        f.write("=" * 80 + "\n\n")
        
        f.write("Average BLEU Scores:\n")
        f.write("-" * 40 + "\n")
        for col in ['BLEU-1', 'BLEU-2', 'BLEU-3', 'BLEU-4']:
            avg = results_df[col].mean()
            f.write(f"{col}: {avg:.4f}\n")
        
        f.write("\n" + "=" * 80 + "\n\n")
        f.write("Detailed Results:\n\n")
        f.write(results_df.to_string())
    
    print(f"✅ Report saved to {filename}")

# Save bulletins
save_translations(bulletin_df, 'public_bulletins_translated.csv')

# Save evaluation report
save_translation_report(results_df, 'translation_evaluation_report.txt')

## 12. Translation Performance Metrics

In [ ]:
import time

def measure_translation_performance(texts: List[str], direction: str = 'en-mr') -> Dict:
    """
    Measure translation performance metrics.
    """
    start_time = time.time()
    
    translations = []
    for text in texts:
        translation = mt_system.translate(text, direction=direction)
        translations.append(translation)
    
    end_time = time.time()
    
    total_time = end_time - start_time
    avg_time = total_time / len(texts)
    total_chars = sum(len(text) for text in texts)
    chars_per_second = total_chars / total_time
    
    return {
        'total_sentences': len(texts),
        'total_time': total_time,
        'avg_time_per_sentence': avg_time,
        'total_characters': total_chars,
        'chars_per_second': chars_per_second,
        'translations': translations
    }

# Test performance
test_sentences = public_bulletins

print("=" * 80)
print("TRANSLATION PERFORMANCE METRICS")
print("=" * 80)

performance = measure_translation_performance(test_sentences)

print(f"\n📊 Performance Statistics:")
print("-" * 40)
print(f"Total Sentences: {performance['total_sentences']}")
print(f"Total Time: {performance['total_time']:.2f} seconds")
print(f"Average Time per Sentence: {performance['avg_time_per_sentence']:.3f} seconds")
print(f"Total Characters: {performance['total_characters']}")
print(f"Characters per Second: {performance['chars_per_second']:.2f}")
print(f"Sentences per Minute: {(60 / performance['avg_time_per_sentence']):.2f}")

## 13. Real-world Use Case: News Article Translation

In [ ]:
# Sample news article
news_article = """
Mumbai Weather Alert: Heavy Rainfall Expected

The India Meteorological Department has issued an orange alert for Mumbai and surrounding areas. 
Heavy to very heavy rainfall is expected in the next 48 hours. 
Citizens are advised to stay indoors and avoid unnecessary travel. 
The municipal corporation has deployed emergency response teams across the city. 
School and colleges will remain closed tomorrow as a precautionary measure. 
For emergency assistance, please call 1077.
"""

print("=" * 80)
print("NEWS ARTICLE TRANSLATION")
print("=" * 80)

print("\n📰 Original English News Article:")
print("-" * 80)
print(news_article)

print("\n⏳ Translating article...")
marathi_article = mt_system.translate_document(news_article, direction='en-mr')

print("\n📰 Translated Marathi News Article:")
print("-" * 80)
print(marathi_article)
print("=" * 80)

## 14. Summary and Recommendations

In [ ]:
print("=" * 80)
print("MACHINE TRANSLATION SYSTEM SUMMARY")
print("=" * 80)

summary = f"""
✅ Translation System Features:
   • Bidirectional translation (English ↔ Marathi)
   • Multiple backends (Google Translate, IndicTrans2)
   • Document and batch translation support
   • BLEU score evaluation
   • Performance metrics tracking

📊 Evaluation Results:
   • Average BLEU-1: {results_df['BLEU-1'].mean():.4f}
   • Average BLEU-2: {results_df['BLEU-2'].mean():.4f}
   • Average BLEU-3: {results_df['BLEU-3'].mean():.4f}
   • Average BLEU-4: {results_df['BLEU-4'].mean():.4f}

⚡ Performance:
   • Average translation time: {performance['avg_time_per_sentence']:.3f} seconds/sentence
   • Processing speed: {performance['chars_per_second']:.2f} characters/second

💡 Applications:
   • Public announcements and notices
   • Government policy documents
   • News articles and media content
   • Educational materials
   • Healthcare information
   • Emergency alerts and warnings

🎯 Recommendations:
   1. Use IndicTrans2 for better accuracy with Indian languages
   2. Validate translations with native speakers for critical content
   3. Implement post-editing for formal documents
   4. Regular model updates for improved performance
   5. Consider domain-specific fine-tuning for specialized content

📁 Output Files Generated:
   • public_bulletins_translated.csv
   • translation_evaluation_report.txt
"""

print(summary)
print("=" * 80)

## 15. Custom Translation Function

In [ ]:
def translate_your_text():
    """
    Quick function to translate your own text.
    """
    print("\n" + "=" * 60)
    print("QUICK TRANSLATION")
    print("=" * 60)
    
    # Get input
    text = input("\nEnter text (English or Marathi): ").strip()
    direction = input("Translation direction (en-mr / mr-en): ").strip().lower()
    
    if direction not in ['en-mr', 'mr-en']:
        print("Invalid direction. Using en-mr.")
        direction = 'en-mr'
    
    # Translate
    result = mt_system.translate(text, direction=direction)
    
    # Display
    print("\n" + "-" * 60)
    print(f"Original: {text}")
    print(f"Translation: {result}")
    print("-" * 60)

# Uncomment to use:
# translate_your_text()

## 🎉 Conclusion

This notebook has demonstrated a comprehensive Machine Translation system for English-Marathi translation with:

1. **Multiple Translation Backends** - Google Translate and IndicTrans2
2. **Bidirectional Translation** - Both English → Marathi and Marathi → English
3. **Quality Evaluation** - BLEU score metrics
4. **Performance Metrics** - Translation speed and efficiency
5. **Real-world Applications** - Public information, news, documents
6. **Batch Processing** - Translate multiple texts efficiently

### Next Steps:
- Fine-tune models on domain-specific data
- Implement human-in-the-loop validation
- Add support for more Indian languages
- Create REST API for production deployment
- Integrate with content management systems